In [21]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [22]:
loader = PyMuPDFLoader(
  file_path = "../data/GST-Acts-and-Rules-Bare-Law-11-04-2025.pdf"
)
documents = loader.load()
documents = documents[20:]
def split_document(documents, chunk_size = 500, chunk_overlap = 100):
  text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = chunk_size,
    chunk_overlap = chunk_overlap,
    separators = ["CHAPTER","Section","Rule","\n\n", "\n", " ", ""]
  )
  split_docs = text_splitter.split_documents(documents)
  
  return split_docs
chunks = split_document(documents)


In [23]:
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder
from typing import List, Tuple, Any, Dict
from openai import OpenAI
from dotenv import load_dotenv
import numpy as np
import uuid
import os

In [24]:
class EmbeddingManger:
  def __init__(self, model_name = "sentence-transformers/all-MiniLM-L12-v2"):
    self.model_name = model_name
    self.model = None
    self.load_model()

  def load_model(self):
    try:
      print(f"Loading model: {self.model_name}")
      self.model = SentenceTransformer(self.model_name)
      print("Model Loaded Successfully.")
      
      
    except Exception as e:
      print(f"Error loading the model {self.model_name}:{e}")
      raise

  def generate_embeddings(self, texts: List[str]) ->np.ndarray:
    if not self.model:
      raise ValueError("Error in Loading the model.")
    
    #print("Generating Embeddings.")
    embeddings = self.model.encode(texts,)
    
    return embeddings

embedding_manager = EmbeddingManger()
#embedding_manager.generate_embeddings(["hello"])


Loading model: sentence-transformers/all-MiniLM-L12-v2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2653.24it/s]


Model Loaded Successfully.


In [25]:
class VectorStore:

  def __init__(self, collection_name = "Pdf_documents", persist_directory ="../chroma" ):
    self.collection_name = collection_name 
    self.persist_directory = persist_directory
    self.client = None
    self.collection = None
    self.initialize_store()

  def initialize_store(self):
    try:
        os.makedirs(self.persist_directory, exist_ok=True)
        self.client = chromadb.PersistentClient(
            path=self.persist_directory)
        try:
            self.client.delete_collection(self.collection_name)
            print("Old collection deleted")
        except:
            print("No old collection found")

        self.collection = self.client.create_collection(
            name=self.collection_name,
            metadata={
                "description": "PDF document embedding for RAG"
            }
        )

        print(f"Collection: {self.collection.name}")
        
    except Exception as e:
        print(e)

  def add_documents(self, embeddings, documents:List[Any]):

    if len(documents) != len(embeddings):
      raise ValueError("no.of documents must match the embeddings")
    
    print(f"Adding {len(documents)} documents to the vector store")

    ids = []
    metadatas = []
    documents_text = []
    embeddings_list = []

    for i, (doc, embedding) in enumerate(zip(documents, embeddings)):

      doc_id = f"doc_{i}"
      ids.append(doc_id)

      metadata = dict(doc.metadata)
      metadata['doc_index'] = i 
      metadata['content_length'] = len(doc.page_content)
      metadatas.append(metadata)

      documents_text.append(doc.page_content)

      embeddings_list.append(embedding.tolist())

    try:
      self.collection.add(
        ids = ids,
        embeddings = embeddings_list,
        metadatas = metadatas,
        documents = documents_text
      )

    except Exception as e:
      print(f"Error adding documents to vector store: {e}")

vector_store = VectorStore()
vector_store

Old collection deleted
Collection: Pdf_documents


In [26]:
texts = [doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(embeddings, chunks)

Adding 3972 documents to the vector store


In [27]:
class RAGRetriever:
  def __init__(self, vector_store:VectorStore, embedding_manager: EmbeddingManger):
    self.vector_store = vector_store
    self.embedding_manager = embedding_manager

  def retrieve(self, query: str, top_k: int = 10, score_threshold: float= 0.0):
    #print(f"Retrieving documents for query: {query}")
    #print(f"Top_k: {top_k}, Score Threshold: {score_threshold}")
    
    query_embedding = self.embedding_manager.generate_embeddings([query])[0]

    if self.vector_store.collection is None:
      raise ValueError("Collection not initialized")
    
    try:
      results = self.vector_store.collection.query(
        query_embeddings = [query_embedding.tolist()], 
        n_results = top_k
      )
      
      retrieved_docs = []

      if results['documents'] and results['documents'][0]:
        documents = results['documents'][0]
        metadatas = results['metadatas'][0]
        distances = results['distances'][0]
        ids = results['ids'][0]

        for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
          similarity_score = 1-distance

          if similarity_score >= score_threshold:
            retrieved_docs.append({
              'id':doc_id,
              'content':document,
              'metadata':metadata,
              'similarity_score':similarity_score,
              'distance':distance,
              'rank':i+1 
            })

        #print(f"Retrieved {len(retrieved_docs)} documents")
      else:
        print("No documents found")

      return retrieved_docs
    
    except Exception as e:
      print(f"Error during retrieval{e}: ")
      return[]


rag_retriever = RAGRetriever(vector_store, embedding_manager)
rag_retriever


In [28]:
class ReRanker:
  def __init__(self,model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"):
    self.model_name = model_name
    self.model = CrossEncoder(self.model_name)
      
  def rerank(self, query, retriever, top_k = 3):
    retrieved_docs = retriever.retrieve(query, top_k = top_k)
    pairs = [
      (query, doc["content"]) for doc in retrieved_docs
    ]
    scores = self.model.predict(pairs)

    for doc, score in zip(retrieved_docs, scores):
      doc['rerank_score'] = float(score)
    

    retrieved_docs.sort(
      key=lambda x:x["rerank_score"], 
      reverse=True
      )
    reranked_docs = retrieved_docs[ : top_k]

    return reranked_docs

re_ranker = ReRanker()
re_ranker

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6683.90it/s]


In [29]:
test_questions = [
    "What is reverse charge?",
    "What is a return?",
    "What is a tax invoice?",
    "What is a non-resident taxable person?",
    "What is Input Tax Credit?",
    "What are blocked credits?",
    "Who is liable for GST registration?",
    "What is GST REG-01?",
    "What is GSTR-1?",
    "What is GSTR-3B?",
    "What is a composite supply?",
    "What is a mixed supply?",
    "What is an exempt supply?",
    "What is tax deduction at source?",
    "What is tax collection at source?",
    "What is an e-way bill?",
    "What is an advance ruling?",
    "What is an assessment under GST?",
    "What is cancellation of registration?",
    "What is revocation of cancellation?"
]

In [30]:
for question in test_questions:
    print("\n")
    print("Question:", question)

    docs = re_ranker.rerank(question, rag_retriever)

    for i, doc in enumerate(docs):
        print(f"\nChunk {i+1}")
        print(doc["content"][:300])



Question: What is reverse charge?

Chunk 1
acting on behalf of such supplier; or 
(b)  collection of the goods by the recipient thereof or by any other person acting on behalf 
of such recipient; 
(97)  "return" means any return prescribed or otherwise required to be furnished by or under this 
Act or the rules made thereunder; 
(98)  "rever

Chunk 2
Rule 37. Reversal of input tax credit in the case of non-payment of consideration.- 
9[(1) A registered person, who has availed of input tax credit on any inward supply of goods or 
services or both, other than the supplies on which tax is payable on reverse charge basis, but fails 
to pay to the su


Question: What is a return?

Chunk 1
CHAPTER VIII : RETURNS 
1[

Chunk 2
CHAPTER IX: RETURNS

Chunk 3
per the return referred to in the said sub-section."


Question: What is a tax invoice?

Chunk 1
Section 31. Tax Invoice.- 
(1)  A registered person supplying taxable goods shall, before or at the time of,- 
(a)  removal of goods for supply

In [42]:
load_dotenv(".env")
llm = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key = os.getenv("API_KEY")
)

def sample_rag(query, retriever, llm):
    results = re_ranker.rerank(query, retriever)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    
    if not results :
        return "No relevant documents found to answer the question."

    prompt = f"""Use the following context to answer the question:
    Context:
    {context}
    Question: {query}
    Answer:"""
    
    response = llm.chat.completions.create(
      model = "nvidia/nemotron-3-nano-30b-a3b",
      messages = [
        {
          "role":"user",
          "content":prompt
        }
      ]
    )
    return response.choices[0].message.content


In [43]:
answer = sample_rag(" What is reverse charge?", rag_retriever, llm)
print("Answer:", answer)

Answer: **Reverse charge** means that the liability to pay tax on a supply of goods or services (or both) falls on the recipient of that supply instead of on the supplier, as provided under the relevant sub‑section of the Act.
